# Kaggle Local LLM Runner

This notebook is designed for Kaggle GPU sessions where internet is disabled.
It launches a local OpenAI-compatible server on `127.0.0.1`, writes a local `.env`, and runs the ARC agent with `--agent=localllm`.

Before running:
- Put this repo somewhere under `/kaggle/working/` or `/kaggle/input/`
- Make sure your model weights are already available locally
- Make sure the ARC local environments are available offline

In [ ]:
from pathlib import Path

def detect_repo_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path('/kaggle/working/ARC-AGI-3-Agents'),
        Path('/kaggle/working'),
    ]
    for base in candidates:
        if (base / 'main.py').exists() and (base / 'agents').exists():
            return base
    for child in Path('/kaggle/working').glob('*'):
        if child.is_dir() and (child / 'main.py').exists() and (child / 'agents').exists():
            return child
    raise FileNotFoundError('Could not auto-detect repo dir under /kaggle/working')

# Edit these values for your Kaggle session.
REPO_DIR = detect_repo_dir()
MODEL_PATH = Path('/kaggle/input/your-model-dir')
SERVED_MODEL_NAME = 'local-arc-model'
GAME_ID = 'locksmith'
AGENT_NAME = 'localllm'
VLLM_PORT = 8000
MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.9
MAX_FRAME_CHARS = 2500
VLLM_TENSOR_PARALLEL_SIZE = 1

assert REPO_DIR.exists(), f'Repo not found: {REPO_DIR}'
assert MODEL_PATH.exists(), f'Model path not found: {MODEL_PATH}'
print('repo:', REPO_DIR)
print('model:', MODEL_PATH)

In [ ]:
import os
os.chdir(REPO_DIR)
print('cwd:', Path.cwd())

## Optional sanity checks

If your Kaggle image already has `uv` and `vllm`, this cell should pass.
If not, install from local wheels or a Kaggle dataset before continuing.

In [ ]:
import shutil
import subprocess

for tool in ['python', 'uv', 'curl']:
    print(tool, '->', shutil.which(tool))

try:
    out = subprocess.run(['python', '-m', 'vllm.entrypoints.openai.api_server', '--help'], check=True, capture_output=True, text=True)
    print('vllm api_server is available')
except Exception as exc:
    print('vllm api_server check failed:', exc)

## Write `.env` for offline local execution

In [ ]:
from textwrap import dedent

env_text = dedent(f'''
DEBUG=False
RECORDINGS_DIR=recordings
SCHEME=http
HOST=127.0.0.1
PORT={VLLM_PORT}
ARC_BASE_URL=https://three.arcprize.org/
OPERATION_MODE=local
ENVIRONMENTS_DIR=environment_files
OPENAI_BASE_URL=http://127.0.0.1:{VLLM_PORT}/v1
LOCAL_LLM_API_KEY=local-dev
LOCAL_LLM_MODEL={SERVED_MODEL_NAME}
LLM_API_STYLE=json
LLM_SUPPORTS_REASONING_EFFORT=False
LLM_MAX_FRAME_CHARS={MAX_FRAME_CHARS}
AGENTOPS_API_KEY=
ARC_API_KEY=
''').strip() + '\n'

Path('.env').write_text(env_text)
print(Path('.env').read_text())

## Launch local vLLM server

This starts an OpenAI-compatible HTTP server on `127.0.0.1`.
If you use a different backend like `llama.cpp` or `Ollama`, skip this cell and make sure `.env` points at that local endpoint.

In [ ]:
import subprocess
from pathlib import Path

server_log = Path('vllm_server.log')
server_cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--host', '127.0.0.1',
    '--port', str(VLLM_PORT),
    '--model', str(MODEL_PATH),
    '--served-model-name', SERVED_MODEL_NAME,
    '--max-model-len', str(MAX_MODEL_LEN),
    '--gpu-memory-utilization', str(GPU_MEMORY_UTILIZATION),
    '--tensor-parallel-size', str(VLLM_TENSOR_PARALLEL_SIZE),
]

with server_log.open('w') as log_fp:
    server_proc = subprocess.Popen(server_cmd, stdout=log_fp, stderr=subprocess.STDOUT)

print('started pid:', server_proc.pid)
print('log:', server_log)

In [ ]:
import json
import time
import urllib.request

models_url = f'http://127.0.0.1:{VLLM_PORT}/v1/models'
last_error = None

for attempt in range(60):
    try:
        with urllib.request.urlopen(models_url, timeout=5) as resp:
            payload = json.loads(resp.read().decode('utf-8'))
        print('server ready on attempt', attempt + 1)
        print(payload)
        break
    except Exception as exc:
        last_error = exc
        time.sleep(5)
else:
    raise RuntimeError(f'vLLM server did not become ready: {last_error}')

## Run the local agent

In [ ]:
import subprocess

run_cmd = ['uv', 'run', 'main.py', '--agent', AGENT_NAME, '--game', GAME_ID]
print('running:', ' '.join(run_cmd))
result = subprocess.run(run_cmd, text=True, capture_output=True)
print('returncode:', result.returncode)
print(result.stdout)
if result.stderr:
    print('--- stderr ---')
    print(result.stderr)

## Inspect logs

In [ ]:
from pathlib import Path

for path in [Path('logs.log'), Path('vllm_server.log')]:
    print(f'===== {path} =====')
    if path.exists():
        text = path.read_text(errors='ignore')
        print(text[-12000:])
    else:
        print('missing')

## Optional cleanup

In [ ]:
if 'server_proc' in globals() and server_proc.poll() is None:
    server_proc.terminate()
    print('terminated pid', server_proc.pid)
else:
    print('server process not running')